# 37 Dressed-State Optimization (v2)

How dressed-state splitting shifts the dominant coherence regime.

**Builds on:** `31_resonance_management.ipynb`

## Central question

> How does dressed-state splitting shift the dominant coherence regime?

## Working statement

Dressed-state splitting shifts the operating regime from magnetic-noise sensitivity toward orbital-state and phononic channels, creating a useful coherence regime between those interactions.

This notebook develops a resonance-management model of the SiV roadmap presented in the seminar.


## 1. Setup

We use a normalized dressed-state splitting:

\[
s \in [0, 1]
\]

where:

- lower `s` represents the magnetic-noise regime
- intermediate `s` represents the useful coherence regime
- higher `s` represents stronger orbital-state and phononic channels

The model captures the regime structure suggested by the seminar roadmap.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)

plt.rcParams["figure.dpi"] = 120


## 2. Resonance-management model

The model has two main interaction-channel terms.

### Magnetic-noise channel

Magnetic-noise sensitivity decreases as dressed-state splitting increases.

### Orbital-state and phononic channels

Orbital-state and phononic interactions become more important at higher dressed-state splitting.

### Combined interaction strength

\[
I_{total}(s) = I_{magnetic}(s) + I_{orbital/phononic}(s)
\]

### Coherence proxy

\[
T_{proxy}(s) = \frac{1}{I_{total}(s)}
\]


In [ ]:
s = np.linspace(0.05, 1.0, 600)

def magnetic_channel(s, amplitude=1.0, rate=4.2, floor=0.03):
    """Magnetic-noise interaction decreases with dressed-state splitting."""
    return amplitude * np.exp(-rate * s) + floor

def orbital_phononic_channels(s, floor=0.10, threshold=0.50, curvature=3.0):
    """Orbital-state and phononic interactions grow after a higher-splitting threshold."""
    excess = np.maximum(s - threshold, 0)
    return floor + curvature * excess**2

I_magnetic = magnetic_channel(s)
I_orbital_phononic = orbital_phononic_channels(s)
I_total = I_magnetic + I_orbital_phononic
T_proxy = 1 / I_total

opt_idx = np.argmax(T_proxy)
s_opt = s[opt_idx]
T_opt = T_proxy[opt_idx]
I_opt = I_total[opt_idx]

s_opt, T_opt, I_opt


## 3. Interaction-channel curves

The useful coherence regime appears where the total interaction strength is minimized.

At lower splitting, magnetic-noise sensitivity dominates the operating regime.

At higher splitting, orbital-state and phononic channels dominate the operating regime.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(s, I_magnetic, label="Magnetic-noise channel")
ax.plot(s, I_orbital_phononic, label="Orbital-state and phononic channels")
ax.plot(s, I_total, linewidth=2.5, label="Combined interaction strength")

ax.axvline(s_opt, linestyle="--", linewidth=1)
ax.scatter([s_opt], [I_opt], s=70, zorder=5)

ax.text(
    s_opt + 0.025,
    I_opt + 0.08,
    "useful coherence regime",
    fontsize=10,
)

ax.set_title("Dressed-State Splitting Shifts the Dominant Interaction Channel")
ax.set_xlabel("Dressed-state splitting, normalized")
ax.set_ylabel("Relative interaction strength")
ax.legend()
ax.grid(True, alpha=0.3)

output = FIGURES / "37_interaction_channels_vs_splitting.png"
plt.savefig(output, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {output}")


## 4. Coherence proxy

The coherence proxy increases as magnetic-noise sensitivity is reduced, then decreases as higher-splitting interaction channels dominate.

The maximum marks the model's useful coherence regime.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(s, T_proxy, linewidth=2.5)
ax.axvline(s_opt, linestyle="--", linewidth=1)
ax.scatter([s_opt], [T_opt], s=80, zorder=5)

ax.text(
    s_opt + 0.025,
    T_opt * 0.92,
    f"useful regime near s = {s_opt:.2f}",
    fontsize=10,
)

ax.set_title("Coherence Regime vs Dressed-State Splitting")
ax.set_xlabel("Dressed-state splitting, normalized")
ax.set_ylabel("Coherence proxy, arbitrary units")
ax.grid(True, alpha=0.3)

output = FIGURES / "37_coherence_regime.png"
plt.savefig(output, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {output}")


## 5. Dominant-regime classification

Classify each splitting value by the dominant interaction channel.

This turns the resonance-management model into a regime map.


In [ ]:
dominant = np.where(
    I_magnetic > I_orbital_phononic,
    "Magnetic-noise regime",
    "Orbital-state and phononic regime",
)

# Add a useful middle band near the coherence proxy maximum.
useful_band = np.abs(s - s_opt) < 0.08
dominant_with_middle = dominant.copy()
dominant_with_middle[useful_band] = "Useful coherence regime"

regime_df = pd.DataFrame(
    {
        "splitting": s,
        "magnetic_noise_channel": I_magnetic,
        "orbital_state_phononic_channels": I_orbital_phononic,
        "combined_interaction_strength": I_total,
        "coherence_proxy": T_proxy,
        "dominant_regime": dominant_with_middle,
    }
)

regime_df.head()


## 6. Dominant-regime map

The regime map is the main design-space output of this notebook.

It translates the continuous model into three operating regimes:

1. magnetic-noise regime
2. useful coherence regime
3. orbital-state and phononic regime


In [ ]:
regime_order = {
    "Magnetic-noise regime": 0,
    "Useful coherence regime": 1,
    "Orbital-state and phononic regime": 2,
}

regime_values = np.array([regime_order[r] for r in dominant_with_middle])

fig, ax = plt.subplots(figsize=(10, 2.6))

ax.imshow(
    regime_values[np.newaxis, :],
    aspect="auto",
    extent=[s.min(), s.max(), 0, 1],
)

ax.axvline(s_opt, linestyle="--", linewidth=1)

ax.set_yticks([])
ax.set_xlabel("Dressed-state splitting, normalized")
ax.set_title("Dominant-Regime Map")

ax.text(0.18, 0.5, "magnetic-noise\nregime", ha="center", va="center", fontsize=10)
ax.text(s_opt, 0.5, "useful\ncoherence", ha="center", va="center", fontsize=10, fontweight="bold")
ax.text(0.82, 0.5, "orbital-state and\nphononic regime", ha="center", va="center", fontsize=10)

output = FIGURES / "37_dominant_regime_map.png"
plt.savefig(output, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {output}")


## 7. Parameter sensitivity

The location of the useful coherence regime depends on how quickly magnetic sensitivity decreases and how quickly higher-splitting interaction channels grow.

This cell sweeps two qualitative parameters:

- magnetic-noise suppression rate
- orbital-state / phononic channel curvature

The output shows how the useful operating point shifts.


In [ ]:
rates = [2.5, 3.5, 4.5, 5.5]
curvatures = [1.5, 2.5, 3.5, 4.5]

rows = []

for rate in rates:
    for curvature in curvatures:
        Im = magnetic_channel(s, rate=rate)
        Iop = orbital_phononic_channels(s, curvature=curvature)
        It = Im + Iop
        Tp = 1 / It
        idx = np.argmax(Tp)
        rows.append(
            {
                "magnetic_suppression_rate": rate,
                "orbital_phononic_curvature": curvature,
                "useful_splitting": s[idx],
                "max_coherence_proxy": Tp[idx],
            }
        )

sensitivity = pd.DataFrame(rows)
sensitivity


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))

for rate in rates:
    subset = sensitivity[sensitivity["magnetic_suppression_rate"] == rate]
    ax.plot(
        subset["orbital_phononic_curvature"],
        subset["useful_splitting"],
        marker="o",
        label=f"magnetic rate {rate}",
    )

ax.set_title("Useful Splitting Shifts with Interaction Strength")
ax.set_xlabel("Orbital-state / phononic curvature")
ax.set_ylabel("Useful dressed-state splitting")
ax.grid(True, alpha=0.3)
ax.legend()

output = FIGURES / "37_useful_splitting_sensitivity.png"
plt.savefig(output, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {output}")


## 8. Summary table

The resonance-management model gives a compact interpretation of the seminar roadmap.


In [ ]:
summary = pd.DataFrame(
    [
        {
            "Splitting regime": "Lower dressed-state splitting",
            "Dominant interaction channel": "Magnetic-noise sensitivity",
            "Design meaning": "Mechanical dressing begins shifting coherence away from magnetic fluctuations",
        },
        {
            "Splitting regime": "Intermediate dressed-state splitting",
            "Dominant interaction channel": "Balanced interaction channels",
            "Design meaning": "Useful coherence regime",
        },
        {
            "Splitting regime": "Higher dressed-state splitting",
            "Dominant interaction channel": "Orbital-state and phononic channels",
            "Design meaning": "Strain, phononic modes, heating, and nonlinear effects become central",
        },
    ]
)

summary


## 9. Interpretation

This resonance-management model supports the Notebook 31 roadmap.

Dressed-state splitting acts as an organizing variable because it shifts the active coherence regime:

- lower splitting emphasizes magnetic-noise sensitivity
- intermediate splitting supports useful coherence
- higher splitting emphasizes orbital-state and phononic channels

Device-level improvements then determine whether that useful coherence regime can support fast pulses and cavity-compatible operation.


## 10. Next notebook

Notebook 41 can connect the optimized dressed-state regime to device constraints.

Candidate title:

> `41_fast_pulse_bandwidth.ipynb`

Candidate question:

> What pulse bandwidth supports the useful dressed-state regime for noise suppression and cavity-compatible control?

Candidate outputs:

- pulse-duration vs bandwidth relation
- IDT bandwidth schematic
- efficiency / bandwidth tradeoff map
